# Data Mesh notebook

Ez a notebook a WebShop Pro domenjeit data productokra bontja. A teljes lab profil:

```bash
docker compose --profile mesh up -d
```

In [ ]:
import json
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'courses.json').exists())
manifest = json.loads((ROOT / 'webshop-lab' / 'docs' / 'tooling-manifest.json').read_text(encoding='utf-8'))
manifest['course_profiles']['data-mesh']

In [ ]:
data_products = [
    {
        'domain': 'sales',
        'product': 'orders_daily_revenue',
        'owner': 'sales-analytics',
        'contract': {'keys': ['order_id'], 'freshness_hours': 24, 'quality_slo': 0.99},
        'consumers': ['finance', 'marketing', 'leadership'],
    },
    {
        'domain': 'marketing',
        'product': 'events_attribution',
        'owner': 'growth-team',
        'contract': {'keys': ['event_id'], 'freshness_hours': 4, 'quality_slo': 0.97},
        'consumers': ['sales', 'ai-data-engineering'],
    },
    {
        'domain': 'support',
        'product': 'support_knowledge_base',
        'owner': 'customer-care',
        'contract': {'keys': ['document_id'], 'freshness_hours': 12, 'quality_slo': 0.98},
        'consumers': ['agentic-ai', 'rag-evaluation-ai-safety'],
    },
]

for product in data_products:
    print(f"{product['domain']}::{product['product']} -> owner={product['owner']}, freshness={product['contract']['freshness_hours']}h")

In [ ]:
def contract_score(product):
    contract = product['contract']
    has_keys = bool(contract.get('keys'))
    freshness_ok = contract.get('freshness_hours', 999) <= 24
    quality_ok = contract.get('quality_slo', 0) >= 0.97
    return sum([has_keys, freshness_ok, quality_ok]) / 3

scores = {p['product']: contract_score(p) for p in data_products}
scores

Kovetkezo lepes: a `webshop-lab/dbt` modellekbol valodi data product contract YAML-t kesziteni, majd CI-ben ellenorizni a breaking change-eket.